# 🔍 Data Mining — Association Analysis
## Lecture 4 Applied Notebook
**Haydar Kılıç | Artificial Intelligence Engineering**

---

This notebook demonstrates all theoretical concepts from the association analysis lecture with hands-on Python implementations.

### 📋 Contents
1. Basic Concepts (Support, Confidence, Lift)
2. Market Basket Data
3. Apriori Algorithm (from scratch)
4. Rule Generation
5. Maximal and Closed Itemsets
6. FP-Growth (from scratch)
7. Pattern Evaluation Measures (Lift, PS, IS)
8. Simpson's Paradox
9. Categorical & Continuous Attributes
10. Sequential Patterns (GSP)
11. Rare & Negative Patterns


In [ ]:
# ============================================================
# 📦 IMPORT REQUIRED LIBRARIES
# ============================================================
import itertools
from collections import defaultdict, Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['font.family'] = 'DejaVu Sans'

print('✅ All libraries loaded successfully!')


---
## 1️⃣ Basic Concepts

### Market Basket Data

| TID | Items |
|-----|-------|
| 1   | {Bread, Milk} |
| 2   | {Bread, Diapers, Beer, Eggs} |
| 3   | {Milk, Diapers, Beer, Cola} |
| 4   | {Bread, Milk, Diapers, Beer} |
| 5   | {Bread, Milk, Diapers, Cola} |

Each transaction represents a customer's shopping basket.


In [ ]:
# ============================================================
# 📊 LECTURE DATA: Market Basket
# ============================================================

transactions = [
    ['Bread', 'Milk'],
    ['Bread', 'Diapers', 'Beer', 'Eggs'],
    ['Milk', 'Diapers', 'Beer', 'Cola'],
    ['Bread', 'Milk', 'Diapers', 'Beer'],
    ['Bread', 'Milk', 'Diapers', 'Cola']
]

# Create binary (0/1) representation
all_items = sorted(set(item for t in transactions for item in t))
binary_data = []
for t in transactions:
    row = {item: (1 if item in t else 0) for item in all_items}
    binary_data.append(row)

df_binary = pd.DataFrame(binary_data, index=[f'T{i+1}' for i in range(len(transactions))])

print('📋 Transaction Table (Binary Representation):')
print('='*55)
print(df_binary.to_string())
print(f'\n📌 Total number of transactions (N): {len(transactions)}')
print(f'📌 Total number of items (d): {len(all_items)}')
print(f'📌 Items: {all_items}')


In [ ]:
# ============================================================
# 📐 BASIC FORMULAS: Support Count, Support, Confidence
# ============================================================

def sigma(itemset, transactions):
    """
    Support Count σ(X):
    Number of transactions containing itemset X.
    σ(X) = |{ tᵢ | X ⊆ tᵢ, tᵢ ∈ T }|
    """
    count = 0
    for t in transactions:
        if set(itemset).issubset(set(t)):
            count += 1
    return count

def support(itemset, transactions):
    """
    Support s(X): FRACTION of transactions containing X.
    s(X) = σ(X) / N
    """
    return sigma(itemset, transactions) / len(transactions)

def confidence(antecedent, consequent, transactions):
    """
    Confidence c(X → Y): Reliability of prediction of Y given X.
    c(X → Y) = σ(X ∪ Y) / σ(X)
    """
    union = list(antecedent) + list(consequent)
    return sigma(union, transactions) / sigma(antecedent, transactions)

# --- Lecture example: {Milk, Diapers} → {Beer} ---
ant = ['Milk', 'Diapers']
con = ['Beer']

sig_union = sigma(ant + con, transactions)
sig_ant   = sigma(ant, transactions)
N         = len(transactions)

s = sig_union / N
c = sig_union / sig_ant

print('📌 LECTURE EXAMPLE: Rule {Milk, Diapers} → {Beer}')
print('='*45)
print(f'  σ({{Milk, Diapers, Beer}}) = {sig_union}')
print(f'  σ({{Milk, Diapers}})       = {sig_ant}')
print(f'  N                          = {N}')
print(f'  Support    s = {sig_union}/{N} = {s:.2f} ({s*100:.0f}%)')
print(f'  Confidence c = {sig_union}/{sig_ant} = {c:.2f} ({c*100:.0f}%)')
print()

# Support values for all single items
print('📊 Support Values for All Single Items:')
print('-'*35)
for item in all_items:
    s_val   = support([item], transactions)
    sig_val = sigma([item], transactions)
    print(f'  {item:<12} σ={sig_val}  s={s_val:.2f} ({s_val*100:.0f}%)')


In [ ]:
# ============================================================
# 📊 VISUALIZATION: Item Frequencies
# ============================================================

item_counts   = {item: sigma([item], transactions) for item in all_items}
item_supports = {item: support([item], transactions) for item in all_items}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: Support count
colors = ['#2ecc71' if v >= 3 else '#e74c3c' for v in item_counts.values()]
bars = ax1.bar(item_counts.keys(), item_counts.values(), color=colors, edgecolor='black', linewidth=0.8)
ax1.axhline(y=3, color='navy', linestyle='--', linewidth=2, label='minsup count = 3')
ax1.set_title('Item Support Counts (σ)', fontweight='bold')
ax1.set_ylabel('Support Count (σ)')
ax1.set_xlabel('Item')
ax1.legend()
for bar, val in zip(bars, item_counts.values()):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, str(val),
             ha='center', va='bottom', fontweight='bold')

# Right: Support ratio
colors2 = ['#2ecc71' if v >= 0.6 else '#e74c3c' for v in item_supports.values()]
bars2 = ax2.bar(item_supports.keys(), item_supports.values(), color=colors2, edgecolor='black', linewidth=0.8)
ax2.axhline(y=0.6, color='navy', linestyle='--', linewidth=2, label='minsup = 60%')
ax2.set_title('Item Support Ratios (s)', fontweight='bold')
ax2.set_ylabel('Support (s)')
ax2.set_xlabel('Item')
ax2.set_ylim(0, 1.1)
ax2.legend()
for bar, val in zip(bars2, item_supports.values()):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.0%}',
             ha='center', va='bottom', fontweight='bold')

green_patch = mpatches.Patch(color='#2ecc71', label='Frequent (≥ minsup)')
red_patch   = mpatches.Patch(color='#e74c3c', label='Infrequent (< minsup)')
fig.legend(handles=[green_patch, red_patch], loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.05))

plt.suptitle('Market Basket: Item Frequency Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('✅ FREQUENT items for minsup=60%: Bread, Beer, Diapers, Milk')
print('❌ PRUNED items: Cola (σ=2), Eggs (σ=1)')


---
## 2️⃣ Apriori Principle and Anti-Monotonicity

> **Theorem (Apriori Principle):** If an itemset is **frequent**, all of its **subsets must also be frequent**.
>
> **Contrapositive (Pruning):** If `{a,b}` is **not** frequent, all supersets containing `{a,b}` are also **not frequent**.

This property is called **Anti-Monotone**:
$$X \subset Y \Rightarrow f(Y) \leq f(X)$$


In [ ]:
# ============================================================
# 🌳 APRIORI PRINCIPLE: Itemset Lattice Visualization
# ============================================================

def visualize_lattice_pruning():
    """
    Shows the itemset lattice and pruning effect for 3 items (a, b, c).
    If {a,b} is not frequent, supersets (abc) are also pruned.
    """
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 8)
    ax.axis('off')
    ax.set_title('Itemset Lattice Structure and Apriori Pruning\n(Red = pruned infrequent itemsets)',
                 fontsize=13, fontweight='bold')

    # Node positions
    nodes = {
        '∅':    (5.0, 6.8),
        'a':    (2.5, 5.2),
        'b':    (5.0, 5.2),
        'c':    (7.5, 5.2),
        'ab':   (2.0, 3.2),
        'ac':   (5.0, 3.2),
        'bc':   (8.0, 3.2),
        'abc':  (5.0, 1.2),
    }
    # Edges
    edges = [
        ('∅','a'), ('∅','b'), ('∅','c'),
        ('a','ab'), ('a','ac'),
        ('b','ab'), ('b','bc'),
        ('c','ac'), ('c','bc'),
        ('ab','abc'), ('ac','abc'), ('bc','abc')
    ]
    # Pruned (infrequent) nodes
    pruned   = {'ab', 'abc'}
    frequent = {'∅','a','b','c','ac','bc'}

    for (u, v) in edges:
        x1, y1 = nodes[u]
        x2, y2 = nodes[v]
        color = '#e74c3c' if (u in pruned or v in pruned) else '#2c3e50'
        style = '--' if (u in pruned or v in pruned) else '-'
        ax.plot([x1, x2], [y1, y2], color=color, linewidth=2, linestyle=style, zorder=1)

    for node, (x, y) in nodes.items():
        if node in pruned:
            fc, ec, tc = '#e74c3c', '#c0392b', 'white'
        elif node == '∅':
            fc, ec, tc = '#95a5a6', '#7f8c8d', 'white'
        else:
            fc, ec, tc = '#2ecc71', '#27ae60', 'white'
        circle = plt.Circle((x, y), 0.5, color=fc, ec=ec, linewidth=2, zorder=2)
        ax.add_patch(circle)
        ax.text(x, y, node, ha='center', va='center', fontsize=11,
                fontweight='bold', color=tc, zorder=3)

    # Annotation
    ax.text(1.0, 0.5, '💡 {a,b} is not frequent → {abc} is never evaluated!\n   This is Apriori pruning.',
            fontsize=10, color='#c0392b',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='#fdebd0', edgecolor='#e74c3c'))
    green_p = mpatches.Patch(color='#2ecc71', label='Frequent Itemset')
    red_p   = mpatches.Patch(color='#e74c3c', label='Infrequent / Pruned')
    ax.legend(handles=[green_p, red_p], loc='upper right', fontsize=10)
    plt.tight_layout()
    plt.show()

visualize_lattice_pruning()
print('Anti-Monotone Property: X ⊂ Y  ⟹  f(Y) ≤ f(X)')
print('Support pruning enables Apriori to run efficiently!')


---
## 3️⃣ Apriori Algorithm (From Scratch)

**Algorithm 4.1 — Frequent Itemset Generation:**
1. Start with k=1; find all frequent single items
2. Generate k-itemset candidates from frequent (k-1)-itemsets (`candidate-gen`)
3. Prune candidates whose subsets are infrequent (`candidate-prune`)
4. Count support from data
5. Repeat until Fk = ∅


In [ ]:
# ============================================================
# 🤖 APRIORI ALGORITHM — FULL IMPLEMENTATION
# ============================================================

def apriori_candidate_gen(frequent_prev):
    """
    Fk-1 × Fk-1 Join Method:
    Merge pairs sharing the same first (k-2) items.
    Requires lexicographic ordering.
    """
    candidates = []
    freq_list = sorted([sorted(s) for s in frequent_prev])
    for i in range(len(freq_list)):
        for j in range(i + 1, len(freq_list)):
            A = freq_list[i]
            B = freq_list[j]
            # First k-2 items the same? (join condition)
            if A[:-1] == B[:-1] and A[-1] < B[-1]:
                candidates.append(A + [B[-1]])
    return [frozenset(c) for c in candidates]

def apriori_candidate_prune(candidates, frequent_prev):
    """
    candidate-prune: Remove a candidate if any of its
    (k-1)-subsets is infrequent.
    """
    pruned = []
    freq_set = set(frequent_prev)
    for cand in candidates:
        # Are all (k-1)-sized subsets frequent?
        subsets = [frozenset(s) for s in itertools.combinations(cand, len(cand)-1)]
        if all(s in freq_set for s in subsets):
            pruned.append(cand)
    return pruned

def apriori(transactions, minsup):
    """
    Full Apriori Algorithm.
    Input:
        transactions: list of lists (each list = one transaction)
        minsup: minimum support threshold (ratio, 0-1)
    Output:
        All frequent itemsets and their support values
    """
    N = len(transactions)
    min_count = minsup * N  # required transaction count

    all_items = sorted(set(item for t in transactions for item in t))
    all_frequent = {}
    iteration_log = []

    # --- Step 1: 1-itemsets ---
    F_curr = {}
    for item in all_items:
        count = sigma([item], transactions)
        if count >= min_count:
            F_curr[frozenset([item])] = count / N

    k = 1
    iteration_log.append((k, dict(F_curr)))
    all_frequent.update(F_curr)

    # --- Step 2+: k-itemsets ---
    while F_curr:
        # Generate candidates
        candidates = apriori_candidate_gen(list(F_curr.keys()))
        # Prune
        candidates = apriori_candidate_prune(candidates, list(F_curr.keys()))

        F_next = {}
        for cand in candidates:
            count = sigma(list(cand), transactions)
            if count >= min_count:
                F_next[cand] = count / N

        k += 1
        if F_next:
            iteration_log.append((k, dict(F_next)))
            all_frequent.update(F_next)
        F_curr = F_next

    return all_frequent, iteration_log


# ============================================================
# ▶️  RUN: minsup = 60%
# ============================================================
minsup = 0.6
frequent_itemsets, log = apriori(transactions, minsup)

print(f'🔍 Apriori Algorithm — minsup = {minsup*100:.0f}% (min count = {minsup*len(transactions):.0f})')
print('='*60)

for k, freq in log:
    print(f'\n📦 {k}-itemset Frequent Itemsets:')
    print("  {:<30} {:>15} {:>10}".format("Itemset", "Support Count", "Support"))
    print("  " + "-"*55)
    for itemset, sup in freq.items():
        count = int(sup * len(transactions))
        label = '{' + ', '.join(sorted(itemset)) + '}'
        print(f'  {label:<30} {count:>15}    {sup:.2f} ✅')

print(f'\n📌 Total frequent itemsets found: {len(frequent_itemsets)}')


In [ ]:
# ============================================================
# 📊 VISUALIZATION: Apriori — Candidate vs Frequent Itemset Comparison
# ============================================================

# Brute-force candidate count vs Apriori candidate count
d = len(all_items)  # 6 items
k_values = [1, 2, 3]

brute_force_counts = []
for k in k_values:
    brute_force_counts.append(len(list(itertools.combinations(all_items, k))))

apriori_counts = [len(v) for (k, v) in log]

x = np.arange(len(k_values))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - width/2, brute_force_counts, width, label='Brute Force (All Candidates)',
            color='#e74c3c', alpha=0.85, edgecolor='black')
b2 = ax.bar(x + width/2, apriori_counts[:len(k_values)], width, label='Apriori (Frequent Itemsets)',
            color='#2ecc71', alpha=0.85, edgecolor='black')

for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(int(bar.get_height())), ha='center', va='bottom', fontweight='bold')
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(int(bar.get_height())), ha='center', va='bottom', fontweight='bold')

ax.set_xlabel('k (itemset size)')
ax.set_ylabel('Number of Itemsets')
ax.set_title('Brute Force vs Apriori: Candidate Count Comparison', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'{k}-itemset' for k in k_values])
ax.legend()

total_bf = sum(brute_force_counts)
total_ap = sum(apriori_counts[:len(k_values)])
reduction = (1 - total_ap/total_bf) * 100
ax.text(0.98, 0.95, f'Total: {total_bf} → {total_ap}\n{reduction:.0f}% reduction!',
        transform=ax.transAxes, ha='right', va='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='orange'))

plt.tight_layout()
plt.show()
print(f'💡 Apriori reduced {total_bf} candidates to {total_ap} → {reduction:.0f}% savings!')


---
## 4️⃣ Rule Generation

Each frequent k-itemset generates at most $2^k - 2$ rules.

**Confidence Anti-Monotonicity:**  
If `X → Y-X` does not meet the confidence threshold, then `X̃ → Y-X̃` (X̃ ⊂ X) will not either.

$$c = \frac{\sigma(Y)}{\sigma(X)} \quad , \quad \tilde{X} \subset X \Rightarrow \sigma(\tilde{X}) \geq \sigma(X) \Rightarrow c(\tilde{X} \to ...) \leq c(X \to ...)$$


In [ ]:
# ============================================================
# 📜 RULE GENERATION — ap-genrules
# ============================================================

def generate_rules(frequent_itemsets, transactions, minconf):
    """
    Generates all rules meeting the confidence threshold
    from frequent itemsets.
    ap-genrules approach:
    1. Start with 1-item consequents
    2. Merge high-confidence rules to form longer consequents
    """
    rules = []
    N = len(transactions)

    for itemset in frequent_itemsets:
        if len(itemset) < 2:
            continue  # At least 2 items required

        # All possible antecedent/consequent splits
        for size in range(1, len(itemset)):
            for consequent in itertools.combinations(itemset, size):
                consequent = frozenset(consequent)
                antecedent = itemset - consequent

                if len(antecedent) == 0:
                    continue

                sig_union = sigma(list(itemset),    transactions)
                sig_ant   = sigma(list(antecedent), transactions)
                sig_con   = sigma(list(consequent), transactions)

                if sig_ant == 0:
                    continue

                sup  = sig_union / N
                conf = sig_union / sig_ant

                # Compute Lift
                sup_con = sig_con / N
                sup_ant = sig_ant / N
                lift = sup / (sup_ant * sup_con) if (sup_ant * sup_con) > 0 else 0

                if conf >= minconf:
                    rules.append({
                        'antecedent': antecedent,
                        'consequent': consequent,
                        'support':    round(sup, 3),
                        'confidence': round(conf, 3),
                        'lift':       round(lift, 3)
                    })

    return rules


# ============================================================
# ▶️  RUN: minsup=60%, minconf=70%
# ============================================================
minconf = 0.7
rules = generate_rules(frequent_itemsets, transactions, minconf)

print(f'📋 Association Rules (minsup={minsup*100:.0f}%, minconf={minconf*100:.0f}%)')
print('='*75)
print("  {:<38} {:>8} {:>8} {:>8}".format("Rule", "Support", "Confidence", "Lift"))
print('  ' + '-'*62)

for r in sorted(rules, key=lambda x: -x['confidence']):
    ant_str  = '{' + ', '.join(sorted(r['antecedent'])) + '}'
    con_str  = '{' + ', '.join(sorted(r['consequent'])) + '}'
    rule_str = f'{ant_str} → {con_str}'
    lift_icon = '⬆️' if r['lift'] > 1 else ('⬇️' if r['lift'] < 1 else '➡️')
    print(f'  {rule_str:<38} {r["support"]:>8.2f} {r["confidence"]:>8.2f} {r["lift"]:>6.2f} {lift_icon}')

print(f'\n📌 Total rules found: {len(rules)}')
print('\n💡 Lift > 1: positive association, Lift = 1: independent, Lift < 1: negative association')


In [ ]:
# ============================================================
# 📊 VISUALIZATION: Rule Distribution — Support vs Confidence (Lift color)
# ============================================================

if rules:
    fig, ax = plt.subplots(figsize=(10, 6))

    supports    = [r['support']    for r in rules]
    confidences = [r['confidence'] for r in rules]
    lifts       = [r['lift']       for r in rules]

    sc = ax.scatter(supports, confidences, c=lifts, cmap='RdYlGn',
                    s=200, zorder=3, edgecolors='black', linewidth=0.8,
                    vmin=0.5, vmax=max(lifts) + 0.5)

    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('Lift Value', fontsize=11)

    # Rule labels
    for r in rules:
        ant = '{' + ','.join(sorted(r['antecedent'])) + '}'
        con = '{' + ','.join(sorted(r['consequent'])) + '}'
        ax.annotate(f'{ant}→{con}', (r['support'], r['confidence']),
                    textcoords='offset points', xytext=(5, 5), fontsize=7.5)

    ax.axhline(minconf,  color='navy',      linestyle='--', linewidth=1.5, label=f'minconf={minconf*100:.0f}%')
    ax.axvline(minsup,   color='darkgreen', linestyle='--', linewidth=1.5, label=f'minsup={minsup*100:.0f}%')

    ax.set_xlabel('Support', fontsize=12)
    ax.set_ylabel('Confidence', fontsize=12)
    ax.set_title('Association Rules: Support vs Confidence\n(Color = Lift value)', fontweight='bold')
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.1)

    plt.tight_layout()
    plt.show()
else:
    print('No rules found for the specified threshold values.')


---
## 5️⃣ Maximal and Closed Itemsets

| Concept | Definition | Property |
|---------|------------|----------|
| **Maximal Frequent** | No superset is frequent | Most compact representation |
| **Closed Frequent** | No superset has the same support | Minimal lossless representation |

Relationship: **Maximal ⊂ Closed Frequent ⊂ All Frequent**


In [ ]:
# ============================================================
# 🔒 MAXIMAL AND CLOSED ITEMSETS
# ============================================================

def find_maximal(frequent_itemsets, transactions):
    """
    Maximal frequent itemsets: frequent itemsets with no frequent superset.
    """
    maximal = []
    freq_keys = list(frequent_itemsets.keys())
    for fs in freq_keys:
        is_maximal = True
        for other in freq_keys:
            if fs < other:  # is fs a proper subset of other?
                is_maximal = False
                break
        if is_maximal:
            maximal.append(fs)
    return maximal

def find_closed(frequent_itemsets, transactions):
    """
    Closed frequent itemsets: no superset has the same support.
    If a superset of X has the same support as X,
    every transaction containing X also contains that superset → X is redundant.
    """
    closed = []
    freq_keys = list(frequent_itemsets.keys())
    for fs in freq_keys:
        sup_fs = frequent_itemsets[fs]
        is_closed = True
        for other in freq_keys:
            if fs < other and abs(frequent_itemsets[other] - sup_fs) < 1e-9:
                is_closed = False
                break
        if is_closed:
            closed.append(fs)
    return closed


maximal  = find_maximal(frequent_itemsets, transactions)
closed   = find_closed(frequent_itemsets, transactions)
all_freq = list(frequent_itemsets.keys())

print('📊 Frequent Itemset Types Comparison')
print('='*55)

print(f'\n🔵 ALL frequent itemsets ({len(all_freq)} total):')
for fs in all_freq:
    label = '{' + ', '.join(sorted(fs)) + '}'
    sup   = frequent_itemsets[fs]
    tags  = []
    if fs in maximal: tags.append('MAXIMAL')
    if fs in closed:  tags.append('CLOSED')
    tag_str = ' [' + ', '.join(tags) + ']' if tags else ''
    print(f'   {label:<28} s={sup:.2f}{tag_str}')

print(f'\n🟡 Closed frequent itemsets ({len(closed)} total):')
for fs in closed:
    label = '{' + ', '.join(sorted(fs)) + '}'
    print(f'   {label:<28} s={frequent_itemsets[fs]:.2f}')

print(f'\n🔴 Maximal frequent itemsets ({len(maximal)} total):')
for fs in maximal:
    label = '{' + ', '.join(sorted(fs)) + '}'
    print(f'   {label:<28} s={frequent_itemsets[fs]:.2f}')

print(f'\n💡 Relationship: |Maximal|={len(maximal)} ≤ |Closed|={len(closed)} ≤ |All Frequent|={len(all_freq)}')


In [ ]:
# ============================================================
# 📊 VISUALIZATION: Nested circle (Venn) diagram
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_title('Frequent Itemset Hierarchy', fontweight='bold', fontsize=13)

# Ellipses
ellipse3 = mpatches.Ellipse((5, 3.5), 9, 5.5, fill=True,
                              facecolor='#d6eaf8', edgecolor='#2980b9', linewidth=2)
ellipse2 = mpatches.Ellipse((5, 3.5), 6.5, 3.8, fill=True,
                              facecolor='#d5f5e3', edgecolor='#27ae60', linewidth=2)
ellipse1 = mpatches.Ellipse((5, 3.5), 3.5, 2, fill=True,
                              facecolor='#fadbd8', edgecolor='#e74c3c', linewidth=2)

ax.add_patch(ellipse3)
ax.add_patch(ellipse2)
ax.add_patch(ellipse1)

ax.text(5, 3.5, f'Maximal\n({len(maximal)} itemsets)', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#922b21')
ax.text(5, 5.2, f'Closed Frequent\n({len(closed)} itemsets)', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#1e8449')
ax.text(1.5, 1.0, f'All Frequent Itemsets\n({len(all_freq)} itemsets)', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#1a5276')

plt.tight_layout()
plt.show()


---
## 6️⃣ FP-Growth Algorithm

FP-Growth encodes the data in a compact structure called an **FP-tree** (prefix tree) and extracts frequent itemsets **without generating candidates**.

| | Apriori | FP-Growth |
|--|---------|----------|
| Database scans | Many | **2** |
| Candidate generation | Yes | **No** |
| Dense data | Slow | **Fast** |


In [ ]:
# ============================================================
# 🌲 FP-TREE (FP-Growth Tree) — FROM-SCRATCH IMPLEMENTATION
# ============================================================

class FPNode:
    """A single node in the FP-Tree."""
    def __init__(self, item, count=0, parent=None):
        self.item     = item      # Item label
        self.count    = count     # Counter
        self.parent   = parent    # Parent node
        self.children = {}        # Child nodes {item: FPNode}
        self.link     = None      # Next node with same item (header table link)

    def increment(self, count=1):
        self.count += count


class FPTree:
    """Frequent Pattern Tree for FP-Growth."""

    def __init__(self, transactions, minsup):
        self.minsup = minsup
        self.N      = len(transactions)
        self.root   = FPNode(None, 0)
        self.header = {}  # {item: (count, first_node)}
        self._build(transactions)

    def _build(self, transactions):
        """Pass 1: count frequencies, Pass 2: build the tree."""
        # Pass 1: Single item frequencies
        item_counts = Counter(item for t in transactions for item in t)
        min_count   = self.minsup * self.N

        # Remove infrequent items, sort descending by frequency
        self.freq_items = {item: cnt for item, cnt in item_counts.items()
                           if cnt >= min_count}
        self.order = {item: -cnt for item, cnt in self.freq_items.items()}

        # Pass 2: Insert each transaction into the tree
        for transaction in transactions:
            # Drop infrequent items, sort descending
            filtered = [item for item in transaction if item in self.freq_items]
            filtered.sort(key=lambda x: self.order[x])
            if filtered:
                self._insert_transaction(filtered)

    def _insert_transaction(self, items):
        """Insert a transaction (sorted items) into the tree."""
        node = self.root
        for item in items:
            if item in node.children:
                # Increment existing node
                node.children[item].increment()
            else:
                # Create new node
                new_node = FPNode(item, 1, parent=node)
                node.children[item] = new_node
                # Link to header table
                self._update_header(item, new_node)
            node = node.children[item]

    def _update_header(self, item, node):
        """Update the header table (linked list)."""
        if item not in self.header:
            self.header[item] = [self.freq_items[item], node]
        else:
            # Append to end of chain
            current = self.header[item][1]
            while current.link is not None:
                current = current.link
            current.link = node

    def print_tree(self, node=None, indent=0, prefix=''):
        """Print the tree in text format."""
        if node is None:
            node = self.root
            print('NULL (root)')
        for child_item, child_node in sorted(node.children.items()):
            print(' ' * indent + prefix + f'{child_item}:{child_node.count}')
            self.print_tree(child_node, indent + 4, '|-- ')


# ============================================================
# ▶️  LECTURE DATA (FP-Tree Example)
# ============================================================
fp_transactions = [
    ['Bread', 'Milk'],
    ['Bread', 'Diapers', 'Beer', 'Eggs'],
    ['Milk', 'Diapers', 'Beer', 'Cola'],
    ['Bread', 'Milk', 'Diapers', 'Beer'],
    ['Bread', 'Milk', 'Diapers', 'Cola']
]

fp_tree = FPTree(fp_transactions, minsup=0.6)

print('🌲 FP-Tree Structure (minsup=60%):')
print('='*40)
fp_tree.print_tree()

print('\n📋 Header Table (frequent items):')
print("  {:<14} {:>10}".format("Item", "Frequency"))
print('  ' + '-'*24)
for item, (count, _) in sorted(fp_tree.header.items(),
                                key=lambda x: -x[1][0]):
    print(f'  {item:<14} {count:>10}')


In [ ]:
# ============================================================
# 🌲 FP-GROWTH: Frequent Itemset Mining via Conditional FP-Trees
# ============================================================

def get_prefix_paths(tree, item):
    """
    Find prefix paths for a specific item.
    These paths are used to build the conditional FP-tree.
    """
    paths = []
    if item not in tree.header:
        return paths
    node = tree.header[item][1]
    while node is not None:
        path   = []
        parent = node.parent
        while parent and parent.item is not None:
            path.append(parent.item)
            parent = parent.parent
        if path:
            paths.append((list(reversed(path)), node.count))
        node = node.link
    return paths

def fp_growth(transactions, minsup):
    """
    FP-Growth: Frequent itemset mining via divide-and-conquer.
    Starts with the least frequent item, builds conditional trees.
    """
    tree      = FPTree(transactions, minsup)
    min_count = minsup * len(transactions)
    frequent  = {}

    def mine(tree, prefix):
        # Process header table in ascending frequency order
        items = sorted(tree.header.keys(), key=lambda x: tree.header[x][0])
        for item in items:
            new_prefix     = prefix | frozenset([item])
            support_count  = tree.header[item][0]
            frequent[new_prefix] = support_count / len(transactions)

            # Build conditional pattern base
            prefix_paths = get_prefix_paths(tree, item)
            if not prefix_paths:
                continue

            # Build conditional transactions
            cond_transactions = []
            for path, count in prefix_paths:
                cond_transactions.extend([path] * count)

            if cond_transactions:
                cond_tree = FPTree(cond_transactions, minsup)
                if cond_tree.header:  # Continue if conditional tree is non-empty
                    mine(cond_tree, new_prefix)

    mine(tree, frozenset())
    return frequent


# ============================================================
# ▶️  RUN AND COMPARE
# ============================================================
fp_results = fp_growth(fp_transactions, minsup=0.6)

print('🌲 FP-Growth Results (minsup=60%):')
print('='*50)
print("  {:<35} {:>8}".format("Itemset", "Support"))
print('  ' + '-'*43)
for fs, sup in sorted(fp_results.items(), key=lambda x: (len(x[0]), -x[1])):
    label = '{' + ', '.join(sorted(fs)) + '}'
    print(f'  {label:<35} {sup:.2f}')

print(f'\n📌 Total frequent itemsets: {len(fp_results)}')

# Compare with Apriori
apriori_results, _ = apriori(fp_transactions, minsup=0.6)
print(f'📌 Apriori result (same data): {len(apriori_results)} itemsets')

# Check set equality
fp_sets = set(fp_results.keys())
ap_sets = set(apriori_results.keys())
if fp_sets == ap_sets:
    print('\n✅ FP-Growth and Apriori produced IDENTICAL results!')
else:
    diff = fp_sets.symmetric_difference(ap_sets)
    print(f'⚠️ Difference found: {diff}')


---
## 7️⃣ Pattern Evaluation Measures

Confidence alone is not sufficient! It ignores `s(Y)`.

| Measure | Formula | Interpretation |
|---------|---------|----------------|
| **Lift** | $\\frac{s(A,B)}{s(A) \\cdot s(B)}$ | >1: positive, =1: independent, <1: negative |
| **PS** | $s(A,B) - s(A) \\cdot s(B)$ | 0: independent |
| **ϕ-correlation** | Normalized PS | [-1, +1] |
| **IS (Cosine)** | $\\frac{s(A,B)}{\\sqrt{s(A) \\cdot s(B)}}$ | [0, 1] |


In [ ]:
# ============================================================
# 📏 OBJECTIVE INTEREST MEASURES
# ============================================================

def compute_measures(A, B, transactions):
    """
    Compute all interest measures for itemsets A and B.
    """
    N    = len(transactions)
    s_A  = sigma(A, transactions) / N
    s_B  = sigma(B, transactions) / N
    s_AB = sigma(A + B, transactions) / N

    # Confidence
    conf_AB = s_AB / s_A if s_A > 0 else 0

    # Lift (Interest Factor)
    lift = s_AB / (s_A * s_B) if (s_A * s_B) > 0 else 0

    # PS (Piatetsky-Shapiro)
    ps = s_AB - s_A * s_B

    # φ-Correlation
    denom_phi = (s_A * (1 - s_A) * s_B * (1 - s_B)) ** 0.5
    phi = (s_AB - s_A * s_B) / denom_phi if denom_phi > 0 else 0

    # IS (Cosine / Jaccard-like)
    is_val = s_AB / (s_A * s_B) ** 0.5 if (s_A * s_B) > 0 else 0

    return {
        's(A)': s_A, 's(B)': s_B, 's(A,B)': s_AB,
        'Confidence': conf_AB, 'Lift': lift,
        'PS': ps, 'ϕ': phi, 'IS': is_val
    }


# ============================================================
# EXAMPLE 1 — False Positive (Tea → Coffee)
# Confidence 75% but coffee is already 80% frequent!
# ============================================================
print('📌 EXAMPLE 1: Tea → Coffee (False Positive)')
print('Contingency table:')

# f11=150, f10=50, f01=650, f00=150  (Tea + Coffee)
N_ex = 1000
f11  = 150   # Tea and Coffee together
f10  = 50    # Tea present, Coffee absent
f01  = 650   # Tea absent, Coffee present
f00  = 150   # Both absent

s_A_ex  = (f11 + f10) / N_ex  # s(Tea)
s_B_ex  = (f11 + f01) / N_ex  # s(Coffee)
s_AB_ex = f11 / N_ex

conf_ex = s_AB_ex / s_A_ex
lift_ex = s_AB_ex / (s_A_ex * s_B_ex)
ps_ex   = s_AB_ex - s_A_ex * s_B_ex

data_table = pd.DataFrame({'Coffee (1)': [f11, f01, f11+f01],
                            'Coffee (0)': [f10, f00, f10+f00],
                            'Total':      [f11+f10, f01+f00, N_ex]},
                           index=['Tea (1)', 'Tea (0)', 'Total'])
print(data_table.to_string())
print(f'\n  s(Tea)  = {s_A_ex:.2f}  s(Coffee) = {s_B_ex:.2f}  s(Tea,Coffee) = {s_AB_ex:.2f}')
print(f'  Confidence = {conf_ex:.2f}  ← Looks high!')
print(f'  Lift       = {lift_ex:.2f}  ← Below 1 → NEGATIVE association!')
print(f'  PS         = {ps_ex:.3f} ← Negative')
print(f'  ⚠️  Coffee is already bought {s_B_ex*100:.0f}% of the time. Tea REDUCES coffee purchase!')

print()

# EXAMPLE 2 — False Negative (Tea → Honey)
print('📌 EXAMPLE 2: Tea → Honey (False Negative)')
f11b = 100;  f10b = 100;  f01b = 20;  f00b = 780
s_A_b  = (f11b + f10b) / N_ex
s_B_b  = (f11b + f01b) / N_ex
s_AB_b = f11b / N_ex
conf_b = s_AB_b / s_A_b
lift_b = s_AB_b / (s_A_b * s_B_b)

print(f'  s(Tea) = {s_A_b:.2f}  s(Honey) = {s_B_b:.2f}  s(Tea,Honey) = {s_AB_b:.2f}')
print(f'  Confidence = {conf_b:.2f}  ← Only 50%, fails minconf!')
print(f'  Lift       = {lift_b:.2f}  ← Above 1 → POSITIVE association (strong!)')
print(f'  ✅ Buying tea greatly increases honey usage beyond {s_B_b*100:.0f}% baseline!')


In [ ]:
# ============================================================
# 📊 VISUALIZATION: Interpreting Lift
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

scenarios = [
    {'title': 'Negative Association\nTea → Coffee', 'lift': lift_ex, 'conf': conf_ex,
     'color': '#e74c3c', 'note': f'Lift={lift_ex:.2f} < 1\nConfidence misleading!'},
    {'title': 'Independent\n(Example)', 'lift': 1.0, 'conf': 0.6,
     'color': '#95a5a6', 'note': 'Lift=1.0\nIndependent'},
    {'title': 'Positive Association\nTea → Honey', 'lift': lift_b, 'conf': conf_b,
     'color': '#2ecc71', 'note': f'Lift={lift_b:.2f} > 1\nStrong association!'}
]

for ax, sc in zip(axes, scenarios):
    bars = ax.bar(['Confidence', 'Lift'], [sc['conf'], sc['lift']],
                  color=[sc['color'], sc['color']], alpha=0.8, edgecolor='black')
    ax.axhline(1, color='black', linestyle='--', linewidth=1.5, label='Independence line (Lift=1)')
    ax.set_title(sc['title'], fontweight='bold')
    ax.set_ylim(0, max(4, sc['lift'] + 0.5))
    for bar, val in zip(bars, [sc['conf'], sc['lift']]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{val:.2f}', ha='center', fontweight='bold')
    ax.text(0.5, 0.95, sc['note'], transform=ax.transAxes,
            ha='center', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
    ax.legend(fontsize=8)

plt.suptitle('Confidence vs Lift: Misleading Cases', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 8️⃣ Simpson's Paradox

> Due to hidden (confounding) variables, the relationship between two variables may appear to go in one direction in **aggregated data**, while appearing different — or even reversed — in **stratified data**.


In [ ]:
# ============================================================
# 🤯 SIMPSON'S PARADOX — HDTV & Exercise Equipment Example
# ============================================================

# Lecture data: Purchasing HDTV vs Exercise Equipment
data = {
    'Student': {
        'Bought HDTV':    {'Bought Exercise': 10,  'No Exercise': 90},   # 10%
        'No HDTV':        {'Bought Exercise': 50,  'No Exercise': 373},  # ~11.8%
    },
    'Employee': {
        'Bought HDTV':    {'Bought Exercise': 200, 'No Exercise': 146},  # ~57.7%
        'No HDTV':        {'Bought Exercise': 60,  'No Exercise': 44},   # ~57.7% ≈ 58.1%
    }
}

print("🔍 Simpson's Paradox Analysis: HDTV → Exercise Equipment")
print('='*60)

total_hdtv_exercise  = 0
total_hdtv           = 0
total_nhdtv_exercise = 0
total_nhdtv          = 0

for group, d in data.items():
    h_e   = d['Bought HDTV']['Bought Exercise']
    h_ne  = d['Bought HDTV']['No Exercise']
    nh_e  = d['No HDTV']['Bought Exercise']
    nh_ne = d['No HDTV']['No Exercise']

    h_total  = h_e + h_ne
    nh_total = nh_e + nh_ne

    r_hdtv  = h_e  / h_total
    r_nhdtv = nh_e / nh_total

    total_hdtv_exercise  += h_e
    total_hdtv           += h_total
    total_nhdtv_exercise += nh_e
    total_nhdtv          += nh_total

    comp = '⬆️  (HDTV → more exercise)' if r_hdtv > r_nhdtv else '⬇️  (HDTV → less exercise)'
    print(f'\n  👥 Group: {group}')
    print(f'     Bought HDTV    : {h_e}/{h_total}  = {r_hdtv*100:.1f}%')
    print(f'     No HDTV        : {nh_e}/{nh_total} = {r_nhdtv*100:.1f}%')
    print(f'     → {comp}')

# Aggregated data
r_combined_hdtv  = total_hdtv_exercise  / total_hdtv
r_combined_nhdtv = total_nhdtv_exercise / total_nhdtv

print(f'\n  📊 AGGREGATED DATA (All groups):')
print(f'     Bought HDTV : {total_hdtv_exercise}/{total_hdtv}  = {r_combined_hdtv*100:.1f}%')
print(f'     No HDTV     : {total_nhdtv_exercise}/{total_nhdtv} = {r_combined_nhdtv*100:.1f}%')
if r_combined_hdtv > r_combined_nhdtv:
    print(f'     → ⬆️  AGGREGATE: HDTV buyers purchase more exercise equipment!')

print()
print('🚨 PARADOX!')
print('   In EVERY group, HDTV buyers purchase LESS exercise equipment.')
print('   But in the aggregated data, the OPPOSITE appears!')
print('   Why? Employees tend to buy both HDTV and exercise equipment.')
print('\n💡 Solution: Identify the hidden variable (customer type) and stratify the data!')


In [ ]:
# ============================================================
# 📊 VISUALIZATION: Simpson's Paradox
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

groups      = ['Student', 'Employee', 'Aggregated']
hdtv_rates  = [10/100, 200/346, r_combined_hdtv]
nhdtv_rates = [50/423, 60/104, r_combined_nhdtv]
colors_hdtv  = ['#3498db'] * 3
colors_nhdtv = ['#e67e22'] * 3

for i, (ax, grp, r_h, r_n) in enumerate(zip(axes, groups, hdtv_rates, nhdtv_rates)):
    bars = ax.bar(['Bought HDTV', 'No HDTV'], [r_h, r_n],
                  color=[colors_hdtv[i], colors_nhdtv[i]],
                  edgecolor='black', alpha=0.85)
    for bar, val in zip(bars, [r_h, r_n]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val*100:.1f}%', ha='center', fontweight='bold', fontsize=11)

    ax.set_title(f'Group: {grp}', fontweight='bold')
    ax.set_ylabel('Exercise Equipment Purchase Rate')
    ax.set_ylim(0, 0.85)

    if i < 2:  # Stratified
        direction   = '⬇️ HDTV buys less' if r_h < r_n else '⬆️ HDTV buys more'
        color_note  = '#e74c3c' if r_h < r_n else '#27ae60'
    else:  # Aggregated
        direction  = '⬆️ AGGREGATE: HDTV buys more!'
        color_note = '#e74c3c'
    ax.text(0.5, 0.95, direction, transform=ax.transAxes,
            ha='center', va='top', fontsize=10, color=color_note,
            bbox=dict(boxstyle='round', facecolor='white', edgecolor=color_note))

plt.suptitle("Simpson's Paradox: Stratified vs Aggregated Data\n"
             'HDTV buys less exercise in every group — but aggregate says opposite!',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 9️⃣ Categorical & Continuous Attributes

Real data is not always binary. Transformation is required for categorical and continuous attributes.


In [ ]:
# ============================================================
# 🏷️ CATEGORICAL ATTRIBUTES: Binarization
# ============================================================

# Internet survey data
internet_data = pd.DataFrame({
    'ID':              [1, 2, 3, 4, 5, 6, 7, 8],
    'Gender':          ['Male','Female','Male','Female','Male','Female','Male','Female'],
    'EducationLevel':  ['Undergrad','Grad','HighSchool','Undergrad','Grad','HighSchool','Undergrad','Undergrad'],
    'HomeComputer':    ['Yes','Yes','Yes','No','Yes','Yes','No','Yes'],
    'OnlineShopping':  ['Yes','No','Yes','Yes','No','Yes','Yes','No'],
    'PrivacyConcern':  ['Yes','Yes','No','Yes','Yes','No','Yes','No']
})

print('📋 Original Survey Data:')
print(internet_data.to_string(index=False))

# --- Binarization ---
# Each attribute-value pair → new binary column
binary_cols = {}

# Categorical columns: one column per distinct value
for col in ['Gender', 'EducationLevel']:
    for val in internet_data[col].unique():
        binary_cols[f'{col}={val}'] = (internet_data[col] == val).astype(int)

# Binary columns: keep only 'Yes'
for col in ['HomeComputer', 'OnlineShopping', 'PrivacyConcern']:
    binary_cols[f'{col}=Yes'] = (internet_data[col] == 'Yes').astype(int)

df_bin = pd.DataFrame(binary_cols)
df_bin.index = internet_data['ID']
df_bin.index.name = 'ID'

print('\n📋 Binary (0/1) Representation:')
print(df_bin.to_string())
print(f'\n✅ {len(internet_data.columns)-1} attributes → {len(df_bin.columns)} binary items')


In [ ]:
# ============================================================
# 📊 CONTINUOUS ATTRIBUTES: Discretization-Based Method
# ============================================================

# Age and chat usage data
np.random.seed(42)
n = 200

# Different chat tendencies for different age groups
ages_young = np.random.randint(16, 26, size=80)
ages_mid   = np.random.randint(26, 44, size=80)
ages_old   = np.random.randint(44, 64, size=40)

chat_young = np.random.choice([1, 0], size=80, p=[0.82, 0.18])  # Lecture example: 81.5%
chat_mid   = np.random.choice([1, 0], size=80, p=[0.45, 0.55])
chat_old   = np.random.choice([1, 0], size=40, p=[0.30, 0.70])  # Lecture example: 30%

ages = np.concatenate([ages_young, ages_mid, ages_old])
chat = np.concatenate([chat_young, chat_mid, chat_old])

df_age = pd.DataFrame({'Age': ages, 'ChatUsage': chat})

# Discretization: 4-year intervals
bins_4 = list(range(16, 68, 4))
df_age['Interval_4yr'] = pd.cut(df_age['Age'], bins=bins_4, right=False)

# Discretization: wide intervals (8 years)
bins_8 = list(range(16, 68, 8))
df_age['Interval_8yr'] = pd.cut(df_age['Age'], bins=bins_8, right=False)

# Chat rate per interval
chat_by_4 = df_age.groupby('Interval_4yr')['ChatUsage'].agg(['mean', 'count'])
chat_by_8 = df_age.groupby('Interval_8yr')['ChatUsage'].agg(['mean', 'count'])

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, chat_by, title, bins in zip(
        axes,
        [chat_by_4, chat_by_8],
        ['4-Year Intervals (Optimal)', '8-Year Intervals (Too Wide)'],
        [bins_4, bins_8]):

    labels = [str(iv) for iv in chat_by.index]
    means  = chat_by['mean'].values
    counts = chat_by['count'].values

    bar_colors = ['#2ecc71' if m >= 0.7 else ('#f39c12' if m >= 0.4 else '#e74c3c')
                  for m in means]
    bars = ax.bar(range(len(labels)), means, color=bar_colors, edgecolor='black', alpha=0.85)

    ax.axhline(0.7, color='navy', linestyle='--', linewidth=2, label='minconf=70%')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Chat Usage Rate')
    ax.set_ylim(0, 1.1)
    ax.set_title(title, fontweight='bold')
    ax.legend()

    for bar, val, cnt in zip(bars, means, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.0%}\n(n={cnt})', ha='center', fontsize=7.5)

plt.suptitle('Discretization-Based Method: Effect of Bin Width Selection\n'
             'Too wide bins → Rules may be lost!',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('💡 Bins too wide → groups merge and meaningful rules disappear!')
print('💡 Bins too narrow → support drops and rules become unseen!')


---
## 🔟 Sequential Patterns (GSP Algorithm)

**Sequence:** $s = \\langle e_1\ e_2\ \\ldots\ e_n \\rangle$, each $e_j$ is an element (transaction), and each $i$ inside is an event.

**Key Differences (from Itemsets):**
- Order matters: $\\langle a, b \\rangle \\neq \\langle b, a \\rangle$
- Events can recur in different elements
- Apriori principle still applies


In [ ]:
# ============================================================
# ⏱️ SEQUENTIAL PATTERN MINING — GSP Algorithm
# ============================================================

def is_subsequence(pattern, sequence):
    """
    Is pattern a subsequence of sequence?
    For each element of pattern, the corresponding element
    must be a subset of the next element in sequence.
    """
    seq_idx = 0
    for pat_element in pattern:
        found = False
        while seq_idx < len(sequence):
            # Is pat_element a subset of sequence[seq_idx]?
            if set(pat_element).issubset(set(sequence[seq_idx])):
                found = True
                seq_idx += 1
                break
            seq_idx += 1
        if not found:
            return False
    return True

def seq_support(pattern, seq_database):
    """Support of a pattern in the sequence database."""
    count = sum(1 for seq in seq_database if is_subsequence(pattern, seq))
    return count / len(seq_database)

def gsp(seq_database, minsup):
    """
    GSP (Generalized Sequential Patterns) Algorithm.
    Applies Apriori logic to sequence data.
    """
    all_items = sorted(set(item for seq in seq_database
                           for element in seq for item in element))
    N = len(seq_database)
    freq_sequences = {}

    # k=1: Single-item sequences
    F_curr = {}
    for item in all_items:
        pat = ((item,),)
        s = seq_support(pat, seq_database)
        if s >= minsup:
            F_curr[pat] = s

    freq_sequences.update(F_curr)

    # k=2: Two-item sequences
    F_next = {}
    items = [p[0][0] for p in F_curr.keys()]
    for i in items:
        for j in items:
            # Separate elements: <{i}{j}>
            pat1 = ((i,), (j,))
            s1 = seq_support(pat1, seq_database)
            if s1 >= minsup:
                F_next[pat1] = s1
            # Same element: <{i,j}> (i < j)
            if i < j:
                pat2 = ((i, j),)
                s2 = seq_support(pat2, seq_database)
                if s2 >= minsup:
                    F_next[pat2] = s2

    freq_sequences.update(F_next)
    return freq_sequences


# ============================================================
# ▶️  WEB PAGE VISIT SEQUENCE EXAMPLE
# ============================================================
# Customers' web page visit sequences
seq_db = [
    [('Homepage',), ('Electronics',), ('Cameras',), ('Order',)],
    [('Homepage',), ('Clothing',), ('Electronics',), ('Order',)],
    [('Electronics',), ('Cameras',), ('Order',)],
    [('Homepage',), ('Electronics',), ('Order',)],
    [('Homepage',), ('Clothing',), ('Order',)],
]

print('📋 Web Visit Sequence Database:')
for i, seq in enumerate(seq_db, 1):
    labels = ' → '.join([e[0] for e in seq])
    print(f'  User {i}: {labels}')

freq_seqs = gsp(seq_db, minsup=0.6)

print(f'\n🔍 Frequent Sequences (minsup=60%):')
print("  {:<40} {:>8}".format("Sequence", "Support"))
print('  ' + '-'*48)
for pat, s in sorted(freq_seqs.items(), key=lambda x: (-len(x[0]), -x[1])):
    if len(pat) == 1:
        label = f'⟨{{"{pat[0][0]}"}}⟩'
    else:
        elements = ['{' + ','.join(f'"{x}"' for x in e) + '}' for e in pat]
        label = '⟨' + ' → '.join(elements) + '⟩'
    print(f'  {label:<40} {s:.2f}')

# Subsequence test
print('\n📌 Subsequence Test:')
seq_test = [('Electronics', 'Beer'), ('Diapers',), ('Beer',)]
s_test   = [('Electronics',), ('Beer',)]
print(f'  s = {seq_test}')
print(f'  ⟨Electronics⟩⟨Beer⟩ is subsequence? {is_subsequence(s_test, seq_test)}')


---
## 1️⃣1️⃣ Rare & Negative Patterns

Sometimes there are **rare but interesting** patterns:
- **Negative correlation:** DVD and VCR together are rare (competing products)
- **h-Confidence (All-Confidence):** Eliminates cross-support patterns


In [ ]:
# ============================================================
# ❌ RARE & NEGATIVE PATTERNS
# ============================================================

def h_confidence(itemset, transactions):
    """
    h-Confidence (All-Confidence):
    h-conf(X) = s(X) / max_i(s(i_i))
    Anti-monotone measure that eliminates cross-support patterns.
    """
    s_X   = support(list(itemset), transactions)
    max_s = max(support([item], transactions) for item in itemset)
    return s_X / max_s if max_s > 0 else 0


# Lecture example: p frequent (83%), q and r rare (17%) but q and r always co-occur!
print('📌 Skewed Support Distribution Problem')
print('='*50)

# p: in 83 transactions, q: 17, r: 17
# q and r always appear together
trans_skewed = []
for _ in range(14):
    trans_skewed.append(['p', 'q', 'r'])   # q and r together
for _ in range(3):
    trans_skewed.append(['p', 'q'])         # p and q
for _ in range(66):
    trans_skewed.append(['p'])              # only p
for _ in range(17):
    trans_skewed.append(['x', 'y'])         # others

N_sk = len(trans_skewed)

s_p  = support(['p'], trans_skewed)
s_q  = support(['q'], trans_skewed)
s_r  = support(['r'], trans_skewed)
s_pq = support(['p', 'q'], trans_skewed)
s_qr = support(['q', 'r'], trans_skewed)

hconf_pq = h_confidence({'p', 'q'}, trans_skewed)
hconf_qr = h_confidence({'q', 'r'}, trans_skewed)

print(f'  Support values: s(p)={s_p:.2f}, s(q)={s_q:.2f}, s(r)={s_r:.2f}')
print(f'  s({{p,q}})={s_pq:.2f}, s({{q,r}})={s_qr:.2f}')

print()
print('  Cross-Support Ratio r(X) = min_support / max_support:')
r_pq = min(s_p, s_q) / max(s_p, s_q)
r_qr = min(s_q, s_r) / max(s_q, s_r)
msg_pq = "⚠️ Cross-support pattern!" if r_pq < 0.5 else "Normal"
print(f'  r({{p,q}}) = {r_pq:.2f}  → {msg_pq}')
msg_qr = "✅ Balanced pattern" if r_qr >= 0.5 else ""
print(f'  r({{q,r}}) = {r_qr:.2f}  → {msg_qr}')

print()
print('  h-Confidence (All-Confidence):')
msg_hpq = "❌ Low: p is very frequent, spurious association!" if hconf_pq < 0.5 else "✅"
print(f'  h-conf({{p,q}}) = {hconf_pq:.2f}  → {msg_hpq}')
msg_hqr = "✅ High: q and r genuinely co-occur!" if hconf_qr >= 0.5 else ""
print(f'  h-conf({{q,r}}) = {hconf_qr:.2f}  → {msg_hqr}')
print()
print('💡 With standard minsup=20%:')
print(f'   {{q,r}} frequent? {"✅ Yes" if s_qr >= 0.2 else "❌ No, missed!"}')
print(f'   {{p,q}} frequent? {"✅ Yes" if s_pq >= 0.2 else "❌ No"}')
print('   ⚠️  h-Confidence makes this distinction!')


In [ ]:
# ============================================================
# ❌➕ NEGATIVELY CORRELATED PRODUCTS (Competing Products)
# ============================================================

print('📌 Negative Correlation: Competing Product Analysis')
print('='*50)

# Example: DVD and VCR are competing products (buying one means not buying the other)
competitor_trans = [
    ['DVD'],
    ['DVD', 'RemoteControl'],
    ['VCR'],
    ['VCR', 'RemoteControl'],
    ['DVD'],
    ['VCR'],
    ['DVD', 'TV'],
    ['VCR', 'TV'],
    ['DVD'],
    ['VCR'],
]

N_c = len(competitor_trans)
items_c = ['DVD', 'VCR', 'RemoteControl', 'TV']

print('  Single item supports:')
for item in items_c:
    s = support([item], competitor_trans)
    print(f'    s({item}) = {s:.2f}')

print('\n  Joint support and interest measures:')
pairs = [(['DVD', 'VCR'], 'Competing'), (['DVD', 'RemoteControl'], 'Complementary')]

for pair, kind in pairs:
    s_pair   = support(pair, competitor_trans)
    s_a      = support([pair[0]], competitor_trans)
    s_b      = support([pair[1]], competitor_trans)
    lift_val = s_pair / (s_a * s_b) if (s_a * s_b) > 0 else 0
    ps_val   = s_pair - s_a * s_b

    neg_corr = s_pair < s_a * s_b
    icon = '⚔️ Negative' if neg_corr else '🤝 Positive'

    print(f'\n  Pair: {pair} ({kind})')
    print(f'    s({pair}) = {s_pair:.2f}')
    print(f'    Lift = {lift_val:.2f}  →  {icon} correlation')
    print(f'    PS   = {ps_val:.3f}')
    if neg_corr:
        print(f'    → Buying one suppresses purchase of the other! (Competing products)')

# Visualization
fig, ax = plt.subplots(figsize=(8, 5))
comparisons = ['DVD vs VCR\n(Competing)', 'DVD vs Remote\n(Complementary)']
lift_vals = [
    support(['DVD', 'VCR'], competitor_trans) /
    (support(['DVD'], competitor_trans) * support(['VCR'], competitor_trans)),
    support(['DVD', 'RemoteControl'], competitor_trans) /
    (support(['DVD'], competitor_trans) * support(['RemoteControl'], competitor_trans))
]
colors_bar = ['#e74c3c' if l < 1 else '#2ecc71' for l in lift_vals]
bars = ax.bar(comparisons, lift_vals, color=colors_bar, edgecolor='black', alpha=0.85, width=0.4)
ax.axhline(1, color='navy', linestyle='--', linewidth=2, label='Independence (Lift=1)')
for bar, val in zip(bars, lift_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'Lift={val:.2f}', ha='center', fontweight='bold', fontsize=12)
ax.set_ylabel('Lift Value')
ax.set_title('Competing vs Complementary Product Relationships', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


---
## 📊 General Summary and Algorithm Comparison


In [ ]:
# ============================================================
# 📊 FINAL: Algorithm Comparison Table
# ============================================================

comparison = pd.DataFrame({
    'Feature': [
        'Database Scans', 'Candidate Generation', 'Memory Usage',
        'Sparse Data', 'Dense Data', 'Ease of Implementation'
    ],
    'Apriori': [
        'Many (k+1 scans)', 'Yes (FₖxFₖ)', 'Moderate',
        '✅ Good', '❌ Slow', '✅ Easy'
    ],
    'FP-Growth': [
        'Only 2', 'None!', 'Structure-dependent',
        '✅ Good', '✅ Fast', '⚠️ Complex'
    ],
    'GSP (Sequential)': [
        'Many', 'Yes', 'Large',
        '✅ Good', '❌ Slow', '⚠️ Moderate'
    ]
})

comparison = comparison.set_index('Feature')
print('📋 Algorithm Comparison Table')
print('='*65)
print(comparison.to_string())

print()
print('='*65)
print('📚 LECTURE CONTENT SUMMARY')
print('='*65)
topics = [
    ('1',  'Basic Concepts',           'Itemset, support, confidence, association rule'),
    ('2',  'Apriori Principle',        'Anti-monotonicity, pruning strategy'),
    ('3',  'Apriori Algorithm',        'candidate-gen, candidate-prune, hash tree'),
    ('4',  'Rule Generation',          'ap-genrules, confidence anti-monotonicity'),
    ('5',  'Compact Representation',   'Maximal and closed frequent itemsets'),
    ('6',  'FP-Growth',                'FP-tree, conditional trees, candidate-free mining'),
    ('7',  'Pattern Evaluation',       'Lift, PS, φ-correlation, IS measures'),
    ("8",  "Simpson's Paradox",        'Hidden variables, stratified analysis'),
    ('9',  'Categorical & Continuous', 'Binarization, discretization, min-Apriori'),
    ('10', 'Sequential Patterns',      'GSP algorithm, timing constraints'),
    ('11', 'Rare & Negative',          'h-confidence, hyperclique, competing products'),
]
for num, topic, desc in topics:
    print(f'  {num:>2}. {topic:<28} → {desc}')

print()
print('✅ Notebook complete!')


---

## 🎯 Exercises

1. Run the Apriori algorithm with `minsup = 0.4`. How many frequent itemsets were found? Compare with the previous result.

2. Manually compute the support, confidence, and lift of the rule `{Bread} → {Milk}`, then verify using the functions.

3. Add a new transaction: `['Bread', 'Milk', 'Beer', 'Diapers']` — which rules' confidence values change?

4. Rebuild the FP-Tree with `minsup = 0.4` and compare the tree structure with the Apriori result.

5. Create data for the classic `{Diapers} → {Beer}` example and compute the lift value.

---
*Data Mining | Association Analysis | Artificial Intelligence Engineering*
